In [1]:
!mkdir -p tests
!mkdir -p .github/workflows

In [2]:
%%writefile app.py
import os
from flask import Flask
import boto3
from botocore.exceptions import ClientError

app = Flask(__name__)

S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://localhost:9000")
AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minio_admin")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "minio_password")
BUCKET_NAME = "vehicle-diagnostic-vault"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1"
)

def ensure_bucket_exists():
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except ClientError:
        try:
            s3.create_bucket(Bucket=BUCKET_NAME)
        except Exception:
            pass

@app.route("/")
def home():
    return "🌐 Cloud-Connected Diagnostic Portal Active"

@app.route("/seed")
def seed_data():
    ensure_bucket_exists()
    report_content = "🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key="live_fault_code.txt",
        Body=report_content
    )
    return "✅ Seeded fault code P0171 into MinIO S3!"

@app.route("/diagnostics")
def diagnostics():
    try:
        ensure_bucket_exists()
        response = s3.get_object(Bucket=BUCKET_NAME, Key="live_fault_code.txt")
        cloud_data = response["Body"].read().decode("utf-8")
        return f"📊 Data Retrieved From Cloud Bucket:{cloud_data}"
    except s3.exceptions.NoSuchKey:
        return "⚠️ No diagnostic data found yet. Visit /seed first or upload a file to MinIO.", 404
    except Exception as e:
        return f"❌ Error connecting to cloud storage:{str(e)}", 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Writing app.py


In [3]:
%%writefile requirements.txt
Flask==3.0.3
boto3==1.34.144
botocore==1.34.144
pytest==8.2.2

Writing requirements.txt


In [4]:
%%writefile tests/test_app.py
import pytest
from app import app

@pytest.fixture
def client():
    app.config['TESTING'] = True
    with app.test_client() as client:
        yield client

def test_home_route(client):
    """Verifica que la ruta principal responda correctamente"""
    response = client.get('/')
    assert response.status_code == 200
    assert b"Cloud-Connected Diagnostic Portal Active" in response.data

Writing tests/test_app.py


In [5]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 5000

CMD ["python", "app.py"]

Writing Dockerfile


In [6]:
%%writefile .github/workflows/deploy.yml
name: CI/CD Pipeline - Next 3

on:
  push:
    branches:
      - main

jobs:
  build-test-push:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout del repositorio
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      - name: Ejecutar pruebas unitarias (CI)
        run: |
          python -m pip install --upgrade pip
          pip install -r containers/ninth_container/repaso/next_3/requirements.txt
          PYTHONPATH=containers/ninth_container/repaso/next_3 pytest containers/ninth_container/repaso/next_3/tests

      - name: Iniciar sesión en Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}

      - name: Configurar Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Construir y publicar imagen (CD)
        uses: docker/build-push-action@v5
        with:
          context: ./containers/ninth_container/repaso/next_3
          push: true
          tags: |
            ${{ secrets.DOCKER_USERNAME }}/engine-s3-service:latest

Writing .github/workflows/deploy.yml


In [7]:
!PYTHONPATH=. pytest tests/

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-9.1.1, pluggy-1.6.0
rootdir: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_3
plugins: anyio-4.4.0
collected 1 item                                                               

tests/test_app.py .                                                      [100%]

============================== 1 passed in 0.60s ===============================


In [8]:
!git add .
!git commit -m "feat: agregar modulo next_3 con Flask, MinIO, pruebas y pipeline CI/CD"
!git push origin main

[main 1cf7ccb] feat: agregar modulo next_3 con Flask, MinIO, pruebas y pipeline CI/CD
 6 files changed, 249 insertions(+), 8 deletions(-)
 rename containers/ninth_container/repaso/next_3/{next_3 => }/.github/workflows/deploy.yml (62%)
 rename containers/ninth_container/repaso/next_3/{next_3 => }/Dockerfile (100%)
 create mode 100644 containers/ninth_container/repaso/next_3/Untitled.ipynb
 rename containers/ninth_container/repaso/next_3/{next_3 => }/app.py (100%)
 rename containers/ninth_container/repaso/next_3/{next_3 => }/requirements.txt (100%)
 rename containers/ninth_container/repaso/next_3/{next_3 => }/tests/test_app.py (100%)
Counting objects: 10, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (8/8), done.
Writing objects: 100% (10/10), 3.10 KiB | 3.10 MiB/s, done.
Total 10 (delta 4), reused 0 (delta 0)
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   5d000cd..1cf7cc

In [11]:
# !docker rm -f mi_microservicio_next3 || true
!docker pull leopinzon75/engine-s3-service:latest
# !docker run -d -p 5001:5000 --name mi_microservicio_next3 leopinzon75/engine-s3-service:latest


latest: Pulling from leopinzon75/engine-s3-service

450697fa: Already exists 
55e0dd95: Already exists 
10ea5bd9: Already exists 
e06c2d45: Already exists 
80dfc3b1: Pulling fs layer 
878ed72b: Pulling fs layer 
f0f30651: Pulling fs layer 
b56a3a36: Pull complete 034kB/5.034kBBADigest: sha256:908819620bb50b3012811b796b1202f8fb3cc0aefff4e6e7ddac9ec3ec611f56
Status: Downloaded newer image for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest


In [12]:
!docker run -d -p 5001:5000 --name mi_microservicio_next3 leopinzon75/engine-s3-service:latest


30c12b5a5e8da81ba8967719f1a9449ead6317d1ebe074cf28985852a8cf8d21


In [ ]:
import requests

print("--- /seed ---")
print(requests.get("http://localhost:5001/seed").text)

print("\n--- /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

In [14]:
!docker logs mi_microservicio_next3

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/urllib3/connection.py", line 203, in _new_conn
🛡️ Connecting to MinIO S3 at: http://localhost:9000
    sock = connection.create_connection(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/botocore/httpsession.py", line 464, in send
    urllib_response = conn.urlopen(
  File "/usr/local/lib/python3.10/site-packages/urllib3/connectionpool.py", line 845, in urlopen
    retries = retries.increment(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/retry.py", line 445, in increment
    raise rera

In [15]:
!docker ps -a | grep mi_microservicio_next3

30c12b5a5e8d   leopinzon75/engine-s3-service:latest   "python app.py"          10 minutes ago   Exited (1) 9 minutes ago                                                    mi_microservicio_next3


In [16]:
# 1. Eliminar cualquier contenedor viejo que haya fallado
!docker rm -f mi_microservicio_next3 || true



mi_microservicio_next3


In [17]:
# 2. Levantar el contenedor mapeando el puerto 5001
!docker run -d -p 5001:5000 --name mi_microservicio_next3 leopinzon75/engine-s3-service:latest

c5b7b2b9ebd3d90329f202a0522aa3cc6ceadaa4aa64572f37588c2df9fdc43e


In [18]:
import requests
import time

# Esperar 2 segundos a que el servidor Flask termine de iniciar dentro del contenedor
time.sleep(2)

try:
    # 1. Probar la ruta principal
    res_home = requests.get("http://localhost:5001/")
    print("--- / (Home) ---")
    print(res_home.text)

    # 2. Probar /seed
    res_seed = requests.get("http://localhost:5001/seed")
    print("\n--- /seed ---")
    print(res_seed.text)

    # 3. Probar /diagnostics
    res_diag = requests.get("http://localhost:5001/diagnostics")
    print("\n--- /diagnostics ---")
    print(res_diag.text)

except Exception as e:
    print(f"❌ Error de conexión: {e}")

❌ Error de conexión: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1125d60d0>: Failed to establish a new connection: [Errno 61] Connection refused'))


In [19]:
!docker logs mi_microservicio_next3


🛡️ Connecting to MinIO S3 at: http://localhost:9000
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/urllib3/connection.py", line 203, in _new_conn
    sock = connection.create_connection(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/botocore/httpsession.py", line 464, in send
    urllib_response = conn.urlopen(
  File "/usr/local/lib/python3.10/site-packages/urllib3/connectionpool.py", line 845, in urlopen
    retries = retries.increment(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/retry.py", line 445, in increment
    raise rera

In [20]:
!docker rm -f mi_microservicio_next3 || true

mi_microservicio_next3


In [21]:
# 1. Eliminar el contenedor que falló
!docker rm -f mi_microservicio_next3 || true



Error: No such container: mi_microservicio_next3


In [22]:
# 2. Levantar el contenedor usando host.docker.internal para llegar a MinIO en tu máquina
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

d5e092ecebdff6053f42dbf2e1e65c15984e218400cc22daa4a26201dfdb5a60


In [23]:
!docker ps | grep mi_microservicio_next3


In [24]:
# 1. Eliminar el contenedor que falló
!docker rm -f mi_microservicio_next3 || true

# 2. Levantar el contenedor usando host.docker.internal para llegar a MinIO en tu máquina
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

mi_microservicio_next3
a4400b2a06c33cc82c343bbecdf61af71b032ffbca84cd9fccbac1ed7196b91a


In [25]:
import requests
import time

time.sleep(2)

# Prueba de inicio
print("--- / (Home) ---")
print(requests.get("http://localhost:5001/").text)

# Prueba de siembra en S3
print("\n--- /seed ---")
print(requests.get("http://localhost:5001/seed").text)

# Prueba de lectura desde S3
print("\n--- /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

--- / (Home) ---


ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1125c6c10>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [26]:
!docker logs mi_microservicio_next3

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/urllib3/connection.py", line 203, in _new_conn
    sock = connection.create_connection(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 85, in create_connection
🛡️ Connecting to MinIO S3 at: http://host.docker.internal:9000
    raise err
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/botocore/httpsession.py", line 464, in send
    urllib_response = conn.urlopen(
  File "/usr/local/lib/python3.10/site-packages/urllib3/connectionpool.py", line 845, in urlopen
    retries = retries.increment(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/retry.py", line 445, in increment
   

In [27]:
!docker network ls


NETWORK ID     NAME             DRIVER    SCOPE
f2ef6cbeb6b8   bridge           bridge    local
6e4e0a86aca2   cinco_default    bridge    local
20e23fda9a60   cuatro_default   bridge    local
03318d8f0b4a   diez_default     bridge    local
8ed6a46f22a0   dos_default      bridge    local
237e064cde9a   host             host      local
057de5fb3f50   none             null      local
44f6bd5969bc   nueve_default    bridge    local
9bdb6e09be55   ocho_default     bridge    local
43bb6cb60883   once_default     bridge    local
b5e5a988fdf4   seis_default     bridge    local
a2626fdf168d   siete_default    bridge    local
1d014b43b8c4   tres_default     bridge    local


In [28]:
!docker rm -f mi_microservicio_next3 || true


mi_microservicio_next3


In [29]:
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://172.17.0.1:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

98661710dbe82d18eace71f7a1af6a7b3dcb29c5879b933a17add20b019eeb69


In [30]:
!docker ps | grep mi_microservicio_next3


In [31]:
!docker logs mi_microservicio_next3

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/urllib3/connection.py", line 203, in _new_conn
🛡️ Connecting to MinIO S3 at: http://172.17.0.1:9000
    sock = connection.create_connection(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/botocore/httpsession.py", line 464, in send
    urllib_response = conn.urlopen(
  File "/usr/local/lib/python3.10/site-packages/urllib3/connectionpool.py", line 845, in urlopen
    retries = retries.increment(
  File "/usr/local/lib/python3.10/site-packages/urllib3/util/retry.py", line 445, in increment
    raise rer

In [35]:
!docker inspect -f '{{range $k, $v := .NetworkSettings.Networks}}{{$k}}{{end}}' $(docker ps -a -q -f name=k8s_minio)

                                                                                  

In [34]:
!docker inspect -f '{{range .NetworkSettings.Networks}}{{.IPAddress}}{{end}}' $(docker ps -a -q -f name=k8s_minio)

{range .NetworkSettings.Networks}{.IPAddress}{end}


In [36]:
# 1. Borrar el contenedor previo
!docker rm -f mi_microservicio_next3 || true



mi_microservicio_next3


In [ ]:
# 2. Levantar el contenedor usando la IP exacta de MinIO en la red de Docker
# (Sustituye 172.18.0.3 por la IP que te dio el paso B arriba)
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://172.18.0.3:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

In [37]:
!docker inspect -f '{{range .NetworkSettings.Networks}}{{.IPAddress}}{{end}}' $(docker ps -a -q -f name=k8s_minio)

{range .NetworkSettings.Networks}{.IPAddress}{end}


In [39]:
import json

# 1. Obtener la información completa de los contenedores de MinIO
minio_info = !docker inspect $(docker ps -q -f name=minio)

if minio_info and minio_info[0].startswith('['):
    data = json.loads("".join(minio_info))
    if data:
        networks = data[0]['NetworkSettings']['Networks']
        print("=== INFORMACIÓN DE MINIO ENCONTRADA ===")
        for net_name, net_data in networks.items():
            print(f"📌 Red de Docker: {net_name}")
            print(f"📌 IP de MinIO en esa red: {net_data['IPAddress']}")
    else:
        print("⚠️ No se encontró ningún contenedor corriendo con el nombre 'minio'.")
else:
    print("⚠️ Revisa si el contenedor de MinIO está encendido con !docker ps")

=== INFORMACIÓN DE MINIO ENCONTRADA ===


In [40]:
# 1. Eliminar el contenedor anterior
!docker rm -f mi_microservicio_next3 || true



Error: No such container: mi_microservicio_next3


In [41]:
# 2. Encender usando la red host
!docker run -d --net=host \
  -e S3_ENDPOINT="http://127.0.0.1:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

af21385f402754a42d6322ce37d109742931079124bbcc189a6e48e104036f56


In [42]:
!docker ps | grep mi_microservicio_next3

In [43]:
import requests
import time

time.sleep(2)

try:
    print("--- 1. Probando /seed ---")
    res_seed = requests.get("http://localhost:5000/seed")
    print(res_seed.text)

    print("\n--- 2. Probando /diagnostics ---")
    res_diag = requests.get("http://localhost:5000/diagnostics")
    print(res_diag.text)

except Exception as e:
    print(f"❌ Error: {e}")

--- 1. Probando /seed ---
❌ Error: HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /seed (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1125b6390>: Failed to establish a new connection: [Errno 61] Connection refused'))


In [44]:
!docker rm -f $(docker ps -aq) 

af21385f4027
36cb7fd07615
bb646d1d452f
6de9bd6b3ee1
3dafe945786a
b02e45bec83d
3e9e77e44a12
d1b54ec06458
0ebce4911aa1
50758b97b123
159a814fa2be
c112086b65e7
c672020167be
678ba04e5354
6dd115c571b0
a58e4c110281
317f853973a8
e3320937e493
c1d64e36e830
8365685aed15


In [45]:
!docker network prune -f

Deleted Networks:
cuatro_default
ocho_default
siete_default
seis_default
diez_default
tres_default
cinco_default
dos_default
once_default
nueve_default



In [46]:
!docker ps -a

CONTAINER ID   IMAGE          COMMAND                  CREATED          STATUS          PORTS     NAMES
4c556c724dce   minio/minio    "/usr/bin/docker-ent…"   41 seconds ago   Up 37 seconds             k8s_minio_minio-deployment-557bb94bbf-7g6zw_default_3ed40980-fa39-45b2-8c15-8fb27f9c034c_2
3567a895d2ec   930f328c3675   "uvicorn main:app --…"   43 seconds ago   Up 38 seconds             k8s_gateway-api_fleet-gateway-deployment-74df6d46b-kt29p_default_84612006-e290-4a07-aba3-2483002d523c_2
cc3ff7c6ea03   e44fc8c61b26   "gunicorn --bind 0.0…"   44 seconds ago   Up 41 seconds             k8s_flask-api_flask-api-deployment-7dff8b4b86-gjckc_default_675fa3eb-4a02-4006-9df0-02d3ad1e64e6_2
69a0178cf942   e44fc8c61b26   "gunicorn --bind 0.0…"   44 seconds ago   Up 41 seconds             k8s_flask-api_flask-api-deployment-7dff8b4b86-7dw5q_default_f64161b2-ad6e-49e7-b243-f48a8ee4f032_2
335ac9edd737   1734d010a179   "python app.py"          45 seconds ago   Up 41 seconds             k8s_flask-web

In [47]:
!kubectl port-forward deployment/minio-deployment 9000:9000 &

OSError: Background processes not supported.

In [48]:
import subprocess
import time

# Iniciar el port-forward de Kubernetes en segundo plano con subprocess
process = subprocess.Popen(
    ["kubectl", "port-forward", "deployment/minio-deployment", "9000:9000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Esperar 2 segundos a que el túnel se establezca
time.sleep(2)
print("✅ Túnel de Kubernetes activado en el puerto 9000.")

✅ Túnel de Kubernetes activado en el puerto 9000.


In [49]:
# 1. Eliminar contenedor anterior si existía
!docker rm -f mi_microservicio_next3 || true

 

Error: No such container: mi_microservicio_next3


In [50]:
# 2. Levantar el microservicio apuntando al MinIO expuesto
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

7e181cdfdf7ecf3a239736c60fe2ac02cadecc57a0db16415eab4e59a165f309


In [51]:
!docker ps | grep mi_microservicio_next3

In [52]:
import requests

# 1. Probar la siembra de datos
res_seed = requests.get("http://localhost:5001/seed")
print("--- /seed ---")
print(res_seed.text)

print("\n" + "="*40 + "\n")

# 2. Probar la lectura de diagnóstico
res_diag = requests.get("http://localhost:5001/diagnostics")
print("--- /diagnostics ---")
print(res_diag.text)

ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: /seed (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1127b4090>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [53]:
!docker logs mi_microservicio_next3

🛡️ Connecting to MinIO S3 at: http://host.docker.internal:9000
📦 Bucket 'engine-trouble-codes' not found. Creating it now...
🚀 Misfire report uploaded to bucket 'engine-trouble-codes'!
📥 Cloud data verification: 100% Match!
💾 Permanent copy stamped to local container logs directory!


In [54]:
import requests

# 1. Probar la ruta principal
print("--- 1. Probar Home ---")
print(requests.get("http://localhost:5001/").text)

# 2. Sembrar datos en S3
print("\n--- 2. Probar /seed ---")
print(requests.get("http://localhost:5001/seed").text)

# 3. Leer diagnóstico desde S3
print("\n--- 3. Probar /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

--- 1. Probar Home ---


ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x11286ded0>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [55]:
import subprocess
import time

# 1. Redirigir el servicio de Flask a tu puerto 5001
subprocess.Popen(
    ["kubectl", "port-forward", "svc/flask-api-service", "5001:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(2)
print("✅ Túnel hacia Flask API activado en http://localhost:5001")

✅ Túnel hacia Flask API activado en http://localhost:5001


In [56]:
!git add .
!git commit -m "fix: mantener servidor Flask activo para recibir peticiones HTTP"
!git push origin main

[main 0061da4] fix: mantener servidor Flask activo para recibir peticiones HTTP
 1 file changed, 2060 insertions(+)
Counting objects: 7, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 25.51 KiB | 4.25 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0)
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   1cf7ccb..0061da4  main -> main


In [57]:
!docker rm -f mi_microservicio_next3 || true
!docker pull leopinzon75/engine-s3-service:latest
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

mi_microservicio_next3
latest: Pulling from leopinzon75/engine-s3-service
Digest: sha256:908819620bb50b3012811b796b1202f8fb3cc0aefff4e6e7ddac9ec3ec611f56
Status: Image is up to date for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest
45e92c072cf1b1e048a7ace8294b7df6f3a08028aea2fc13f84d24269bcc35d5


In [1]:
import os
from pathlib import Path

# Definir la ruta exacta para ver_4 dentro de next_4
target_dir = Path("/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_3/ver_tres_uno")

# Crear la carpeta principal si no existe (sin incluir la subcarpeta logs)
target_dir.mkdir(parents=True, exist_ok=True)

# Cambiar el directorio de trabajo del kernel de Jupyter
os.chdir(target_dir)

# Confirmar dónde estamos ubicados
print("📍 Directorio actual de Jupyter:", os.getcwd())

📍 Directorio actual de Jupyter: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_3/ver_tres_uno
